# Evaluation

150 questions. Retrieval is scored with MRR, nDCG, and keyword coverage. Answers are scored by `gpt-4o-mini` on accuracy, completeness, and relevance, from 1 to 5.

Green starts at 0.9 for MRR and nDCG, and 4.5 for an answer score. Amber starts at 0.75 and 4.0.


## Tests


In [1]:
import sys
from pathlib import Path

from dotenv import load_dotenv

cwd = Path.cwd().resolve()
repo_root = next((p for p in [cwd, *cwd.parents] if (p / ".git").exists()), cwd)
load_dotenv(repo_root / ".env", override=True)
sys.path.insert(0, str(repo_root / "W5-RAG"))

from collections import Counter

from evaluation.eval import evaluate_answer, evaluate_retrieval
from evaluation.test import load_tests

tests = load_tests()
print(len(tests))
print(Counter(test.category for test in tests))
print(tests[0].question)
print(tests[0].keywords)
print(tests[0].reference_answer)


150
Counter({'direct_fact': 70, 'temporal': 20, 'spanning': 20, 'comparative': 10, 'numerical': 10, 'relationship': 10, 'holistic': 10})
Who won the prestigious IIOTY award in 2023?
['Maxine', 'Thompson', 'IIOTY']
Maxine Thompson won the prestigious Insurellm Innovator of the Year (IIOTY) award in 2023.


## One retrieval


In [2]:
example = tests[0]
retrieval = evaluate_retrieval(example)
print(f"MRR {retrieval.mrr:.4f}")
print(f"nDCG {retrieval.ndcg:.4f}")
print(f"keywords {retrieval.keywords_found}/{retrieval.total_keywords}")
print(f"coverage {retrieval.keyword_coverage:.1f}%")


MRR 0.6667
nDCG 0.5561
keywords 2/3
coverage 66.7%


## One answer


In [3]:
score, generated, docs = evaluate_answer(example)
print(generated)
print()
print(score.feedback)
print(f"accuracy {score.accuracy}")
print(f"completeness {score.completeness}")
print(f"relevance {score.relevance}")
print(len(docs), "chunks")


Maxine won the prestigious IIOTY (Insurellm Innovator of the Year) award in 2023.

The generated answer correctly identifies the winner of the IIOTY award but omits the last name and provides an incorrect abbreviation for the award. It is relevant but lacks completeness and accuracy due to these issues.
accuracy 3.0
completeness 3.0
relevance 5.0
10 chunks


## All retrieval scores


In [4]:
rows = [evaluate_retrieval(test) for test in tests]
count = len(rows)
print(f"tests {count}")
print(f"MRR {sum(row.mrr for row in rows) / count:.4f}")
print(f"nDCG {sum(row.ndcg for row in rows) / count:.4f}")
print(f"coverage {sum(row.keyword_coverage for row in rows) / count:.1f}%")


tests 150
MRR 0.7946
nDCG 0.7949
coverage 93.0%


## Page


In [5]:
from app import launch

launch()


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
